In [ ]:
!pip install pandas -q
!pip install seaborn  -q

In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score


### EDA 

In [ ]:
df = pd.read_csv("data/compustat.csv")

print(df.shape)


(620141, 58)


,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,conml,...,xsgaq,capxy,dltisy,dvy,oancfy,oibdpy,scstkcy,mkvaltq,prchq,prclq
0,I,USD,STD,INDL,C,SERV.1,2010-01-31,1082,SERVIDYNE INC,Servidyne Inc,...,2.457,0.211,0.00,0.149,-1.274,-2.603,NaN,6.7252,2.300,1.530
1,I,USD,STD,INDL,C,AIM.1,2010-01-31,1173,AEROSONIC CORP,Aerosonic Corp,...,1.958,1.991,0.00,0.000,-0.032,4.895,NaN,14.7001,6.130,3.750
2,I,USD,STD,INDL,C,ACEL.,2010-01-31,1259,TAMIR BIOTHECHNOLOGY INC,Tamir Biotechnology Inc,...,0.489,0.002,3.25,0.000,-2.692,-1.022,NaN,8.9897,0.301,0.138
3,A,USD,STD,INDL,C,ABM,2010-01-31,1410,ABM INDUSTRIES INC,ABM Industries Inc,...,60.832,7.379,131.00,6.992,-8.913,32.669,NaN,1007.6067,21.650,17.940
4,I,USD,STD,INDL,C,LGTY,2010-01-31,1562,LOGILITY SPPLY CHAIN SOL INC,Logility Supply Chain Solutions Inc,...,8.196,0.418,0.00,6.835,-3.740,8.211,NaN,138.6659,6.750,5.440


In [8]:
df.head()

,costat,curcdq,datafmt,indfmt,consol,tic,datadate,gvkey,conm,conml,...,dltisy,dvy,oancfy,oibdpy,scstkcy,mkvaltq,prchq,prclq,year,quarter
705,A,USD,STD,INDL,C,AIR,2010-02-28,1004,AAR CORP,AAR Corp,...,0.000,0.0,94.647,NaN,NaN,885.0870,26.08,18.77,2010,1
9876,A,USD,STD,INDL,C,AIR,2010-05-31,1004,AAR CORP,AAR Corp,...,63.545,0.0,153.156,NaN,NaN,777.8348,25.91,18.03,2010,2
19004,A,USD,STD,INDL,C,AIR,2010-08-31,1004,AAR CORP,AAR Corp,...,0.000,0.0,7.218,NaN,NaN,609.2237,19.46,14.91,2010,3
28067,A,USD,STD,INDL,C,AIR,2010-11-30,1004,AAR CORP,AAR Corp,...,0.000,0.0,29.812,NaN,NaN,974.4917,24.60,15.56,2010,4
36979,A,USD,STD,INDL,C,AIR,2011-02-28,1004,AAR CORP,AAR Corp,...,0.000,0.0,59.902,NaN,NaN,1084.5592,29.05,24.68,2011,1


In [5]:
print(df.duplicated().sum(), "duplicate rows found and deleted")
df = df.drop_duplicates()

0 duplicate rows found and deleted


In [6]:
df['datadate'] = pd.to_datetime(df['datadate'])

df = df.sort_values(['gvkey', 'datadate'])

# year-quarter index
df['year'] = df['datadate'].dt.year
df['quarter'] = df['datadate'].dt.quarter


In [7]:
obs_per_firm = df.groupby('gvkey')['datadate'].count()

# Keep firms with at least 12 quarters (3 years)
valid_firms = obs_per_firm[obs_per_firm >= 12].index
df = df[df['gvkey'].isin(valid_firms)]


In [6]:
targets = ['oancfy', 'cheq']

df[targets].isna().mean()

oancfy    0.337817
cheq      0.316685
dtype: float64

In [17]:
missing = df.isna().mean().sort_values(ascending=False)

missing.head(15)

findltq    0.997649
udoltq     0.988634
scstkcy    0.973893
deraltq    0.890139
derlltq    0.868834
cshopq     0.684295
xrdq       0.680582
xaccq      0.665951
ipodate    0.584371
dd1q       0.529537
npq        0.475842
lltq       0.474066
drcq       0.473048
ivltq      0.460825
wcapq      0.450105
dtype: float64

### BASELINE